# Agent 指令当做可执行的约束

把指令写成大白话是在许愿。把指令写成约束才是测试。工作台将每条规则变成agent可以在运行时检查、审查者可以针对事实做验证的东西。

## 问题描述

典型的`AGENTS.md`读起来像看板上的文档。它告诉agent要小心、测试要全面、不安全的时候询问。三天后，agent上线了不带测试的改动，写到了禁止的文件夹中，并且从不提问，因为它已经不知道这些内容在哪一行了。

当指令可操作时，是非常强力的。解决方案是将规则写成工作台可以解释，并且审查时可以打分的东西。

## 基本概念

规则属于`doc/agent-rules.md`，离短的根路由远。每条规则有名字、类别以及检查。

```mermaid
flowchart LR
A[AGENTS.md]-->B[doc/agent-rules.md]-->C[rule_checker.py]-->D[rule_repoert.json]-->E[Reviewer]
```

### 大多规则从属的分类

|分类|回答了什么问题|例子|
|---|---|---|
|Startup|工作开始前必须为真的事实|状态文件存在且最新|
|Forbidden|不允许发生的东西|不要编辑xxx|
|Definition of Done|什么能证明任务完成了|pytest通过|
|Uncertainty|agent不清楚时怎么办|问用户而不是猜|
|Approval|什么东西需要人类批准|破坏性行为|

### 规则是机器可读的

每条规则有标题、分类以及一行描述，`check`域内指向`rule_checker.py`中的函数。添加一个规则意味着添加一个检查，检查内容随工作台负责增长。

### 规则是差异友好的

每条规则有一个自己先导的markdown文件。重命名对差异不可见，新的规则也是在所属目录下添加。过时的规则会被删除而不是注释，因为工作台携带的是事实对应的源码，而不是团队之前的感受历史。

### 规则和框架中的护栏

框架护栏是运行时强制执行的规则。本节中的规则是护栏所实现的人类可读的，可被审查的表述。两者你都需要：运行时捕捉一轮中出现的违规，规则集则被用于证明运行时在做正确的事。

### 渐进式披露：一张图、而不是百科全书

`AGENTS.md`不断增长的原因是所有的案例都在往里面添加，而没有人从里面移除。一年以后，这个文件涨到2000行，agent读第一整页的时候，上下文预算就被消耗完了，行动的时候就很少按它被告知的方式去做。一个巨大的指令文件的失败原因跟一份40页的在线文档一样：读它的人扫了一眼，但关注不到真正重要的地方。

修复方案不是更短的文件，而是分层。根级的路由文件保持小，不靠指针以外的方式引用其他内容，让每个会话都能够读取。深层的话题文件只有当任务需要关注到它们的时候才去加载。所以说给agent一张图，而不是整本百科全书，让它自己导航到需要关注的页面。

|层|放在哪里|什么时候读|大小预算|
|---|---|---|---|
|Router|`AGENTS.md`|每个会话，总是读|小于50行|
|Rules|`docs/agent-rules.md`|每个会话，开始的时候|每类一页|
|Topic docs|`docs/<topic>.md`|任务需要关注的时候|按需|

可达性测试：一个agent应该能在最多两次跳跃内摸到任何规则，因此router必须按路径关联每个话题，而不是用大白话描述它。

新鲜度测试：router应该尽可以能短，审查者能够在每个PR重新读它，这是唯一阻止router静默增长的方式。

# 开始编码

对应本章核心：**指令 = 可执行约束（name/category/check）**、**五类规则（Startup/Forbidden/DoD/Uncertainty/Approval）**、**checker → rule_report.json**、**渐进披露（router ≤50 行，两跳可达）**。  
先用玩具注册表跑通检查与分层加载；再用 **LangChain 工具 + DeepSeek** 在写操作前强制跑规则。不硬凑 PyTorch。


## 1. 教学玩具：规则注册表 + 渐进披露

- 每条规则：`id` / `category` / `check` 函数名。
- 检查产出 `RuleVerdict`，汇总成 `rule_report.json` 给审查者。
- Router 只含路径指针；话题文档按需加载（最多两跳）。


In [ ]:
from __future__ import annotations

import json
import re
import shutil
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Literal

Category = Literal["startup", "forbidden", "definition_of_done", "uncertainty", "approval"]


@dataclass
class Rule:
    """机器可读规则：描述给人看，check 给运行时执行。"""

    id: str
    category: Category
    title: str
    description: str
    check: str  # checker 注册表中的函数名


@dataclass
class RuleVerdict:
    """单条规则检查结果。"""

    rule_id: str
    category: Category
    ok: bool
    detail: str


@dataclass
class WorkContext:
    """检查器看到的工作台事实（不是愿望）。"""

    root: Path
    state_exists: bool = False
    state_fresh: bool = True
    edited_paths: list[str] = field(default_factory=list)
    tests_passed: bool | None = None
    asked_user: bool = False
    uncertain: bool = False
    destructive: bool = False
    human_approved: bool = False


Checker = Callable[[WorkContext], tuple[bool, str]]


class RuleChecker:
    """将规则 id → 可执行检查；产出审查者可读报告。"""

    def __init__(self) -> None:
        self._fns: dict[str, Checker] = {}
        self._register_builtins()

    def _register_builtins(self) -> None:
        self._fns["state_file_exists"] = lambda ctx: (
            ctx.state_exists,
            "agent_state.json present" if ctx.state_exists else "missing state file",
        )
        self._fns["state_is_fresh"] = lambda ctx: (
            ctx.state_fresh,
            "state fresh" if ctx.state_fresh else "stale state",
        )
        self._fns["no_forbidden_paths"] = self._no_forbidden
        self._fns["pytest_green"] = lambda ctx: (
            ctx.tests_passed is True,
            "pytest green" if ctx.tests_passed else f"tests={ctx.tests_passed}",
        )
        self._fns["ask_when_uncertain"] = lambda ctx: (
            (not ctx.uncertain) or ctx.asked_user,
            "asked user" if ctx.asked_user or not ctx.uncertain else "guessed under uncertainty",
        )
        self._fns["approval_for_destructive"] = lambda ctx: (
            (not ctx.destructive) or ctx.human_approved,
            "approved" if ctx.human_approved or not ctx.destructive else "destructive without approval",
        )

    def _no_forbidden(self, ctx: WorkContext) -> tuple[bool, str]:
        forbidden_prefix = ("secrets/", ".env", "prod_config/")
        bad = [p for p in ctx.edited_paths if any(p.startswith(f) or f in p for f in forbidden_prefix)]
        return (not bad, "ok" if not bad else f"edited forbidden: {bad}")

    def register(self, name: str, fn: Checker) -> None:
        self._fns[name] = fn

    def run_one(self, rule: Rule, ctx: WorkContext) -> RuleVerdict:
        """
        Args:
            rule: 规则定义。
            ctx: 当前工作事实。

        Returns:
            verdict: 通过与否 + 细节。
        """
        fn = self._fns.get(rule.check)
        if fn is None:
            return RuleVerdict(rule.id, rule.category, False, f"unknown check: {rule.check}")
        ok, detail = fn(ctx)
        return RuleVerdict(rule.id, rule.category, ok, detail)

    def run_all(self, rules: list[Rule], ctx: WorkContext) -> list[RuleVerdict]:
        return [self.run_one(r, ctx) for r in rules]

    def write_report(self, path: Path, verdicts: list[RuleVerdict]) -> dict[str, Any]:
        """
        写出 rule_report.json 供审查者打分。

        Returns:
            report: 汇总 dict。
        """
        report = {
            "passed": all(v.ok for v in verdicts),
            "verdicts": [asdict(v) for v in verdicts],
            "by_category": {},
        }
        for v in verdicts:
            report["by_category"].setdefault(v.category, []).append(v.ok)
        path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
        return report


DEFAULT_RULES: list[Rule] = [
    Rule("R-START-1", "startup", "State exists", "agent_state.json must exist", "state_file_exists"),
    Rule("R-START-2", "startup", "State fresh", "state must be up to date", "state_is_fresh"),
    Rule("R-FORB-1", "forbidden", "No secrets", "do not edit secrets/ or .env", "no_forbidden_paths"),
    Rule("R-DOD-1", "definition_of_done", "Tests pass", "pytest must be green", "pytest_green"),
    Rule("R-UNC-1", "uncertainty", "Ask don't guess", "ask user when uncertain", "ask_when_uncertain"),
    Rule("R-APR-1", "approval", "Destructive needs OK", "human approval for destructive ops", "approval_for_destructive"),
]


@dataclass
class ProgressiveDocs:
    """渐进披露：router → rules → topic，最多两跳。"""

    root: Path

    def write_fixture(self, *, fat_router: bool = False) -> None:
        """写入分层文档夹具。"""
        docs = self.root / "docs"
        docs.mkdir(parents=True, exist_ok=True)
        if fat_router:
            router = "# AGENTS.md\n" + "\n".join(f"- note {i}: " + ("x" * 40) for i in range(80))
        else:
            router = (
                "# AGENTS.md (router)\n\n"
                "- Rules: `docs/agent-rules.md`\n"
                "- Topic auth: `docs/auth.md`\n"
                "- Topic deploy: `docs/deploy.md`\n"
                "- State: `agent_state.json`\n"
            )
        (self.root / "AGENTS.md").write_text(router, encoding="utf-8")
        (docs / "agent-rules.md").write_text(
            "# Rules\n\nSee checks in rule_checker. Categories: startup/forbidden/dod/uncertainty/approval.\n",
            encoding="utf-8",
        )
        (docs / "auth.md").write_text("# Auth topic\nValidators live in src/auth.py\n", encoding="utf-8")
        (docs / "deploy.md").write_text("# Deploy topic\nNeed approval for prod.\n", encoding="utf-8")

    def router_line_count(self) -> int:
        return len((self.root / "AGENTS.md").read_text(encoding="utf-8").splitlines())

    def freshness_ok(self, *, max_lines: int = 50) -> bool:
        """审查者每个 PR 能重读完 router。"""
        return self.router_line_count() <= max_lines

    def resolve(self, topic: str) -> tuple[list[str], str]:
        """
        从 router 最多两跳到达话题文档。

        Returns:
            (hops, content)：跳转路径与文件内容。
        """
        hops: list[str] = ["AGENTS.md"]
        router = (self.root / "AGENTS.md").read_text(encoding="utf-8")
        # 第一跳：从 router 抽路径
        paths = re.findall(r"`([^`]+)`", router)
        target = None
        for p in paths:
            if topic in p or Path(p).stem == topic:
                target = p
                break
        if target is None:
            raise FileNotFoundError(f"topic {topic} not linked from router")
        hops.append(target)
        if len(hops) > 3:  # start + 2 hops
            raise RuntimeError("more than 2 hops")
        content = (self.root / target).read_text(encoding="utf-8")
        return hops, content


print("instruction constraints ready | rules + checker + progressive docs")


## 2. 玩具示例：违规拦截、报告、两跳可达、胖路由失败


In [ ]:
def demo_instruction_constraints() -> None:
    """断言五类检查、报告、渐进披露。"""
    checker = RuleChecker()
    root = Path(tempfile.mkdtemp(prefix="iec_"))
    try:
        # 违规：写了 secrets + 测试失败 + 不确定却没问
        bad_ctx = WorkContext(
            root=root,
            state_exists=True,
            state_fresh=True,
            edited_paths=["secrets/api_key.txt", "src/ok.py"],
            tests_passed=False,
            uncertain=True,
            asked_user=False,
            destructive=True,
            human_approved=False,
        )
        verdicts = checker.run_all(DEFAULT_RULES, bad_ctx)
        by_id = {v.rule_id: v for v in verdicts}
        assert by_id["R-FORB-1"].ok is False
        assert by_id["R-DOD-1"].ok is False
        assert by_id["R-UNC-1"].ok is False
        assert by_id["R-APR-1"].ok is False
        report_path = root / "rule_report.json"
        report = checker.write_report(report_path, verdicts)
        assert report["passed"] is False and report_path.exists()
        print("violations caught + report written")

        # 合规上下文
        good_ctx = WorkContext(
            root=root,
            state_exists=True,
            state_fresh=True,
            edited_paths=["src/auth.py"],
            tests_passed=True,
            uncertain=False,
            destructive=False,
        )
        good = checker.run_all(DEFAULT_RULES, good_ctx)
        assert all(v.ok for v in good)
        print("all categories green on clean context")

        # 渐进披露
        docs = ProgressiveDocs(root)
        docs.write_fixture(fat_router=False)
        assert docs.freshness_ok()
        hops, content = docs.resolve("auth")
        assert hops == ["AGENTS.md", "docs/auth.md"]
        assert len(hops) - 1 <= 2
        assert "Validators" in content
        print("two-hop reachability ok")

        docs.write_fixture(fat_router=True)
        assert docs.freshness_ok() is False
        print("fat router fails freshness test")
        print("TOY DEMO OK")
    finally:
        shutil.rmtree(root, ignore_errors=True)


demo_instruction_constraints()


## 3. 生产级：规则检查工具 + DeepSeek

agent 通过工具读取规则、提交拟编辑路径与测试结果；`run_rule_checks` 写出报告。宣称完成前必须 `passed=true`。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_CHECKER = RuleChecker()
PROD_CTX = WorkContext(root=Path("."), state_exists=True, state_fresh=True)


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_prod_docs() -> str:
    """重建带短路由的演示仓库。"""
    global PROD_ROOT, PROD_CTX
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    PROD_ROOT = Path(tempfile.mkdtemp(prefix="iec_prod_"))
    ProgressiveDocs(PROD_ROOT).write_fixture(fat_router=False)
    (PROD_ROOT / "agent_state.json").write_text("{}", encoding="utf-8")
    PROD_CTX = WorkContext(root=PROD_ROOT, state_exists=True, state_fresh=True)
    return json.dumps({"root": str(PROD_ROOT)}, ensure_ascii=False)


class EditArgs(BaseModel):
    path: str
    replace: bool = Field(False, description="If true, replace edited_paths with only this path")


class FlagArgs(BaseModel):
    uncertain: bool = False
    asked_user: bool = False
    destructive: bool = False
    human_approved: bool = False
    tests_passed: bool | None = None


class TopicArgs(BaseModel):
    topic: str = "auth"


class EmptyArgs(BaseModel):
    pass


def build_rule_tools() -> list[StructuredTool]:
    def _reset(**kwargs: Any) -> str:
        return reset_prod_docs()

    def _list_rules(**kwargs: Any) -> str:
        return json.dumps([asdict(r) for r in DEFAULT_RULES], ensure_ascii=False)

    def _propose_edit(**kwargs: Any) -> str:
        a = EditArgs(**kwargs)
        if a.replace:
            PROD_CTX.edited_paths = [a.path]
        else:
            PROD_CTX.edited_paths.append(a.path)
        return json.dumps({"edited_paths": list(PROD_CTX.edited_paths)}, ensure_ascii=False)

    def _set_flags(**kwargs: Any) -> str:
        a = FlagArgs(**kwargs)
        if a.uncertain:
            PROD_CTX.uncertain = True
        if a.asked_user:
            PROD_CTX.asked_user = True
        if a.destructive:
            PROD_CTX.destructive = True
        if a.human_approved:
            PROD_CTX.human_approved = True
        if a.tests_passed is not None:
            PROD_CTX.tests_passed = a.tests_passed
        return json.dumps(asdict(PROD_CTX) | {"root": str(PROD_CTX.root)}, default=str, ensure_ascii=False)

    def _run_checks(**kwargs: Any) -> str:
        assert PROD_ROOT is not None
        verdicts = PROD_CHECKER.run_all(DEFAULT_RULES, PROD_CTX)
        report = PROD_CHECKER.write_report(PROD_ROOT / "rule_report.json", verdicts)
        return json.dumps(report, ensure_ascii=False)

    def _load_topic(**kwargs: Any) -> str:
        assert PROD_ROOT is not None
        hops, content = ProgressiveDocs(PROD_ROOT).resolve(TopicArgs(**kwargs).topic)
        return json.dumps({"hops": hops, "content": content, "fresh_router": ProgressiveDocs(PROD_ROOT).freshness_ok()}, ensure_ascii=False)

    return [
        StructuredTool.from_function(name="reset_docs", description="Reset short-router fixture repo.", func=_reset, args_schema=EmptyArgs),
        StructuredTool.from_function(name="list_rules", description="List machine-readable rules.", func=_list_rules, args_schema=EmptyArgs),
        StructuredTool.from_function(name="propose_edit", description="Record a path the agent wants to edit.", func=_propose_edit, args_schema=EditArgs),
        StructuredTool.from_function(name="set_flags", description="Set uncertainty/tests/approval flags on work context.", func=_set_flags, args_schema=FlagArgs),
        StructuredTool.from_function(name="run_rule_checks", description="Execute all rule checks and write rule_report.json.", func=_run_checks, args_schema=EmptyArgs),
        StructuredTool.from_function(name="load_topic", description="Progressive disclosure: resolve topic via router hops.", func=_load_topic, args_schema=TopicArgs),
    ]


RULE_TOOLS = build_rule_tools()


def build_control_agent() -> Any:
    system = (
        "You enforce executable instruction constraints.\n"
        "Flow: reset_docs -> list_rules -> propose_edit/set_flags -> run_rule_checks.\n"
        "If report.passed is false, fix flags/paths; never claim done while failed. Chinese."
    )
    return create_agent(get_llm(), RULE_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 700 else str(m.content)[:700] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


def scripted_constraint_flow() -> dict[str, Any]:
    """
    确定性路径：先踩 forbidden，再改合规路径并过检。

    Returns:
        report: 两次检查结果。
    """
    reset_prod_docs()
    PROD_CTX.edited_paths = ["secrets/token"]
    PROD_CTX.tests_passed = False
    bad = PROD_CHECKER.run_all(DEFAULT_RULES, PROD_CTX)
    bad_report = PROD_CHECKER.write_report(PROD_ROOT / "rule_report_bad.json", bad)
    PROD_CTX.edited_paths = ["src/auth.py"]
    PROD_CTX.tests_passed = True
    PROD_CTX.uncertain = False
    PROD_CTX.destructive = False
    good = PROD_CHECKER.run_all(DEFAULT_RULES, PROD_CTX)
    good_report = PROD_CHECKER.write_report(PROD_ROOT / "rule_report.json", good)
    hops, _ = ProgressiveDocs(PROD_ROOT).resolve("auth")
    return {"bad_passed": bad_report["passed"], "good_passed": good_report["passed"], "hops": hops}


print(f"executable constraints production ready | {MODEL}")


## 4. 生产示例：违规报告 → 修复 → passed

无 `DEEPSEEK_API_KEY` 则跳过 LLM agent，仍跑脚本化检查路径。


In [ ]:
def demo_production_constraints() -> None:
    """生产：报告门禁 + 可选 DeepSeek 工具循环。"""
    rep = scripted_constraint_flow()
    print("=== scripted checks ===")
    print(json.dumps(rep, ensure_ascii=False, indent=2))
    assert rep["bad_passed"] is False
    assert rep["good_passed"] is True
    assert rep["hops"][0] == "AGENTS.md"
    print("report gate ok")

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK")
        return

    reset_prod_docs()
    agent = build_control_agent()
    out = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        "reset_docs，list_rules，propose_edit secrets/x，"
                        "set_flags tests_passed=false，run_rule_checks（应失败），"
                        "再 propose_edit path=src/auth.py replace=true、set_flags tests_passed=true，"
                        "run_rule_checks 直到 passed，load_topic auth，中文说明规则如何变成约束。"
                    )
                )
            ]
        }
    )
    print("=== control agent ===")
    print(format_agent_messages(out.get("messages") or [])[:2200])
    final_report = json.loads((PROD_ROOT / "rule_report.json").read_text(encoding="utf-8"))
    assert final_report["passed"] is True
    print("PROD DEMO OK")


demo_production_constraints()
